In [1]:
from notebookutils import mssparkutils
import pandas as pd

files = mssparkutils.fs.ls("Files")  # ← correct path

datasets = []
for f in files:
    if f.name.lower().endswith(".xlsx"):
        datasets.append({
            "dataset_name": f.name.replace(".xlsx", ""),
            "category":     "Auto",
            "active":       True
        })

if datasets:
    spark.createDataFrame(pd.DataFrame(datasets)) \
         .write.mode("overwrite").saveAsTable("dataset_registry")
    print(f"✅ Registered {len(datasets)} datasets:")
    for d in datasets:
        print(f"   → {d['dataset_name']}")
else:
    print("⚠ No .xlsx files found in Lakehouse Files folder")

StatementMeta(, 1b9c8b1e-6ab3-4392-9ab0-408c40dd1aa8, 3, Finished, Available, Finished, False)

✅ Registered 3 datasets:
   → Customer_Master_Data
   → Global_Climate_Records
   → Retail_Observability_Intelligence


In [2]:
try:
    DATASET_NAME = getArgument("DATASET_NAME")
except:
    DATASET_NAME = "Customer_Master_Data"

print(f"Dataset: {DATASET_NAME}")

StatementMeta(, 1b9c8b1e-6ab3-4392-9ab0-408c40dd1aa8, 4, Finished, Available, Finished, False)

Dataset: Customer_Master_Data


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from datetime import datetime

# Remove the lakehouse prefix — use default lakehouse only
BRONZE_TABLE = f"bronze_{DATASET_NAME.lower().replace(' ', '_')}"
SILVER_TABLE = f"silver_{DATASET_NAME.lower().replace(' ', '_')}"
SILVER_TABLE      = f"silver_{DATASET_NAME.lower().replace(' ', '_')}"
RUN_TIMESTAMP     = datetime.now().isoformat()

print(f"Source (Bronze) : {BRONZE_TABLE}")
print(f"Target (Silver) : {SILVER_TABLE}")
print(f"Run time        : {RUN_TIMESTAMP}")

StatementMeta(, 1b9c8b1e-6ab3-4392-9ab0-408c40dd1aa8, 5, Finished, Available, Finished, False)

Source (Bronze) : bronze_customer_master_data
Target (Silver) : silver_customer_master_data
Run time        : 2026-06-04T11:31:31.266557


In [4]:
# Create Bronze table from Excel
import pandas as pd
from notebookutils import mssparkutils

# Copy file from Sales_Intelligence_LH to local temp — works regardless of default lakehouse
src = f"abfss://Sales_Intelligence@onelake.dfs.fabric.microsoft.com/Sales_Intelligence_LH.Lakehouse/Files/{DATASET_NAME}.xlsx"
dst = f"file:/tmp/{DATASET_NAME}.xlsx"

mssparkutils.fs.cp(src, dst)

df = pd.read_excel(f"/tmp/{DATASET_NAME}.xlsx")
df.columns = df.columns.str.lower().str.replace(" ", "_")

spark.createDataFrame(df).write.mode("overwrite").saveAsTable(BRONZE_TABLE)
print(f"✅ Bronze table created: [{BRONZE_TABLE}]")
print(f"   Columns: {df.columns.tolist()}")
print(f"   Rows: {len(df)}")

StatementMeta(, 1b9c8b1e-6ab3-4392-9ab0-408c40dd1aa8, 6, Finished, Available, Finished, False)

✅ Bronze table created: [bronze_customer_master_data]
   Columns: ['customer_id', 'customer_name', 'email', 'phone', 'city', 'country', 'customer_segment', 'registration_date', 'account_status', 'annual_spend']
   Rows: 100


In [5]:
# Read from external lakehouse using 3-part name: lakehouse.schema.table
bronze_df = spark.read.table(BRONZE_TABLE)

# Normalise column names
for col in bronze_df.columns:
    bronze_df = bronze_df.withColumnRenamed(
        col, col.lower().strip().replace(" ", "_")
    )

print(f"Rows loaded from Bronze: {bronze_df.count()}")
print(f"Columns: {bronze_df.columns}")
display(bronze_df.limit(5))

StatementMeta(, 1b9c8b1e-6ab3-4392-9ab0-408c40dd1aa8, 7, Finished, Available, Finished, False)

Rows loaded from Bronze: 100
Columns: ['customer_id', 'customer_name', 'email', 'phone', 'city', 'country', 'customer_segment', 'registration_date', 'account_status', 'annual_spend']


SynapseWidget(Synapse.DataFrame, 6fbe0c69-e8cf-446d-96ac-ed1b57ea26d1)

In [6]:
silver_df = bronze_df

# ── Null handling — works on ALL columns dynamically ──────────
all_cols = silver_df.columns
silver_df = silver_df.withColumn(
    "null_issue_flag",
    F.when(
        F.greatest(*[F.col(c).isNull().cast("int") for c in all_cols]) == 1, 1
    ).otherwise(0)
) if len(all_cols) >= 2 else silver_df.withColumn("null_issue_flag", F.lit(0))

# ── Filter nulls on first column (universal primary key) ──────
first_col = all_cols[0]
silver_df = silver_df.filter(F.col(first_col).isNotNull())

# ── Fill missing values dynamically by type ───────────────────
from pyspark.sql.types import StringType, DoubleType, LongType, IntegerType, DateType

fill_str = {c: "UNKNOWN"     for c, t in silver_df.dtypes if isinstance(spark.range(1).select(F.lit(None).cast(t)).schema[0].dataType, StringType)}
fill_num = {c: 0.0           for c, t in silver_df.dtypes if t in ("double", "float", "int", "bigint", "long")}
silver_df = silver_df.fillna(fill_str)
silver_df = silver_df.fillna(fill_num)

# ── Duplicate detection — uses first column as ID ─────────────
dup_counts = silver_df.groupBy(first_col).count().withColumnRenamed("count", "_dup_count")
silver_df  = silver_df.join(dup_counts, on=first_col, how="left")
silver_df  = silver_df.withColumn(
    "duplicate_flag", F.when(F.col("_dup_count") > 1, 1).otherwise(0)
).drop("_dup_count")

# ── Negative value flag — checks ANY numeric column ───────────
numeric_cols = [c for c, t in silver_df.dtypes if t in ("double", "float", "int", "bigint", "long")
                and c not in ("duplicate_flag", "null_issue_flag")]

if numeric_cols:
    neg_flags = [F.when(F.col(c) < 0, 1).otherwise(0) for c in numeric_cols]
    silver_df = silver_df.withColumn(
        "negative_value_flag",
        F.when(F.greatest(*neg_flags) == 1, 1).otherwise(0)
        if len(neg_flags) >= 2
        else neg_flags[0]
    )
else:
    silver_df = silver_df.withColumn("negative_value_flag", F.lit(0))

# ── Region validation — only if region column exists ──────────
if "region" in silver_df.columns:
    VALID_REGIONS = ["APAC", "EMEA", "NA", "LATAM", "NORTH AMERICA", "SOUTH ASIA"]
    silver_df = silver_df.withColumn(
        "invalid_region_flag",
        F.when(F.upper(F.col("region")).isin(VALID_REGIONS), 0).otherwise(1)
    )
else:
    silver_df = silver_df.withColumn("invalid_region_flag", F.lit(0))

# ── Row severity — universal ───────────────────────────────────
flag_sum = (F.col("null_issue_flag") + F.col("duplicate_flag") +
            F.col("negative_value_flag") + F.col("invalid_region_flag"))

silver_df = silver_df.withColumn(
    "row_severity",
    F.when(flag_sum >= 3, "CRITICAL")
     .when(flag_sum == 2, "HIGH")
     .when(flag_sum == 1, "MEDIUM")
     .otherwise("HEALTHY")
)

# ── Silver metadata ────────────────────────────────────────────
silver_df = silver_df \
    .withColumn("silver_processed_at", F.lit(RUN_TIMESTAMP)) \
    .withColumn("source_dataset",      F.lit(DATASET_NAME))

total   = silver_df.count()
flagged = silver_df.filter(F.col("row_severity") != "HEALTHY").count()
print(f"Dataset     : {DATASET_NAME}")
print(f"Total rows  : {total}")
print(f"Clean rows  : {total - flagged}")
print(f"Flagged rows: {flagged}")
display(silver_df.limit(5))

StatementMeta(, 1b9c8b1e-6ab3-4392-9ab0-408c40dd1aa8, 8, Finished, Available, Finished, False)

Dataset     : Customer_Master_Data
Total rows  : 100
Clean rows  : 92
Flagged rows: 8


SynapseWidget(Synapse.DataFrame, 48f79fc3-895c-40eb-b740-696bdf584afb)

In [7]:
silver_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Sales_Intelligence_LH.dbo." + SILVER_TABLE)

print(f"✅ Silver table written: [Sales_Intelligence_LH.dbo.{SILVER_TABLE}]")

StatementMeta(, 1b9c8b1e-6ab3-4392-9ab0-408c40dd1aa8, 9, Finished, Available, Finished, False)

✅ Silver table written: [Sales_Intelligence_LH.dbo.silver_customer_master_data]


In [2]:
display(spark.table("dataset_registry"))

StatementMeta(, afce202d-01de-4cb7-b51c-5dc8acf42578, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fee85b57-6678-425b-a62a-0b09c3c34844)